<a href="https://colab.research.google.com/github/Nayab189/flyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract


## 1. Unit of analysis + time window

**Unit of Analysis (Grain):**
Each row represents one unique pseudonymized content page (`content_id`) for a specific client tenant at a monthly snapshot, integrating traditional search performance alongside multi-channel AI-assistant referral signals.

**Time Window & Split Logic:**
* **Observation & Feature Window:** We use February 2026 (`2026-02-01` to `2026-02-28`) to build our contract and extract baseline features (GSC performance, AI-assistant referrals, and dimensional attributes).
* **Label & Evaluation Window:** We use March 2026 (`2026-03-01` to `2026-03-31`) to evaluate relative traffic/impression degradation, enforcing true temporal separation and eliminating same-window data leakage.
* **Sealed Test Window:** Per warehouse design standards, the final month (**June 2026**) is treated as an out-of-time sealed test window for final model evaluation.

In [ ]:
import duckdb
from google.colab import userdata

# Safely fetch HF_TOKEN from Colab Secrets
HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

# 1. Advanced Data Contract with Fixed NaN-to-Zero COALESCE and LEFT JOIN Survivorship Bias Fix
df = con.sql(
    f"""
    WITH feature_window AS (
        SELECT
            content_hash_id AS content_id,
            BOOL_OR(gsc_data_available) AS is_available,
            SUM(gsc_impressions) AS impressions_30d,
            SUM(gsc_clicks) AS clicks_30d,
            CASE
                WHEN SUM(gsc_impressions) > 0 THEN (SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions))
                ELSE 0.0
            END AS ctr_30d,
            AVG(gsc_sum_position) AS avg_position,
            COALESCE(SUM(sessions_ai), 0) AS ai_sessions_30d,
            COALESCE(SUM(ai_chatgpt), 0) AS ai_chatgpt_30d,
            COALESCE(SUM(ai_perplexity), 0) AS ai_perplexity_30d,
            COALESCE(SUM(ai_gemini), 0) AS ai_gemini_30d,
            COALESCE(SUM(ai_copilot), 0) AS ai_copilot_30d,
            COALESCE(SUM(ai_claude), 0) AS ai_claude_30d
        FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
        WHERE report_date BETWEEN '2026-02-01' AND '2026-02-28'
          AND gsc_data_available IS TRUE
        GROUP BY content_hash_id
    ),
    label_window AS (
        SELECT
            content_hash_id AS content_id,
            SUM(gsc_clicks) AS future_clicks_30d,
            SUM(gsc_impressions) AS future_impressions_30d
        FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
        WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
        GROUP BY content_hash_id
    )
    SELECT
        f.content_id,
        '2026-02-28' AS snapshot_date,
        f.is_available,
        f.impressions_30d,
        f.clicks_30d,
        f.ctr_30d,
        f.avg_position,
        f.ai_sessions_30d,
        f.ai_chatgpt_30d,
        f.ai_perplexity_30d,
        f.ai_gemini_30d,
        f.ai_copilot_30d,
        f.ai_claude_30d,
        COALESCE(c.word_count, 800) AS word_count,
        COALESCE(DATE_DIFF('day', CAST(c.content_created_date AS DATE), DATE '2026-02-28'), 90) AS content_age_days,

        -- True Early Warning Decay Label with Vanished Page Handling (Survivorship Fix)
        CASE
            WHEN l.content_id IS NULL THEN 1  -- Completely vanished from fact table in March = Extreme Decline
            WHEN l.future_impressions_30d < (f.impressions_30d * 0.8) THEN 1
            ELSE 0
        END AS is_declining

    FROM feature_window f
    LEFT JOIN label_window l ON f.content_id = l.content_id
    LEFT JOIN read_parquet('{rel}/dim_content.parquet') c
        ON f.content_id = c.content_hash_id
    LIMIT 100000
"""
).df()

print("Temporal Separation & Survivorship-Safe Join successfully implemented!")
print("Dataset shape:", df.shape)

# 2. Mandatory AI-Referral Traffic Sparsity Profile Check
ai_sparsity = duckdb.query("""
    SELECT
        COUNT(*) as total_rows,
        SUM(CASE WHEN ai_sessions_30d > 0 THEN 1 ELSE 0 END) as rows_with_any_ai_traffic,
        ROUND(AVG(CASE WHEN ai_sessions_30d > 0 THEN 1.0 ELSE 0.0 END) * 100, 2) as pct_with_ai_traffic,
        AVG(ai_sessions_30d) as mean_ai_sessions,
        MAX(ai_sessions_30d) as max_ai_sessions
    FROM df
""").df()

print("\n--- AI-Referral Traffic Sparsity Profile ---")
print(ai_sparsity)

# 3. Verify Content Age Variance (Addressing Claude's check)
age_check = duckdb.query("""
    SELECT
        COUNT(DISTINCT content_age_days) as unique_age_values,
        MIN(content_age_days) as min_age,
        MAX(content_age_days) as max_age
    FROM df
""").df()
print("\n--- Content Age Variance Check ---")
print(age_check)

df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Temporal Separation & Survivorship-Safe Join successfully implemented!
Dataset shape: (100000, 16)

--- AI-Referral Traffic Sparsity Profile ---
   total_rows  rows_with_any_ai_traffic  pct_with_ai_traffic  \
0      100000                    1524.0                 1.52   

   mean_ai_sessions  max_ai_sessions  
0           0.04762             59.0  

--- Content Age Variance Check ---
   unique_age_values  min_age  max_age
0                221        3      444


,content_id,snapshot_date,is_available,impressions_30d,clicks_30d,ctr_30d,avg_position,ai_sessions_30d,ai_chatgpt_30d,ai_perplexity_30d,ai_gemini_30d,ai_copilot_30d,ai_claude_30d,word_count,content_age_days,is_declining
0,content_04c67f3541177192,2026-02-28,True,246.0,1.0,0.004065,151.464286,0.0,0.0,0.0,0.0,0.0,0.0,3168,144,0
1,content_05acc92c165f4386,2026-02-28,True,137.0,0.0,0.000000,49.037037,0.0,0.0,0.0,0.0,0.0,0.0,4135,144,1
2,content_0b33d8960857ad90,2026-02-28,True,8.0,0.0,0.000000,14.333333,0.0,0.0,0.0,0.0,0.0,0.0,3232,144,1
3,content_0f30e04e709c7b5d,2026-02-28,True,121.0,0.0,0.000000,39.428571,0.0,0.0,0.0,0.0,0.0,0.0,3211,144,0
4,content_1207efddce873942,2026-02-28,True,174.0,0.0,0.000000,194.000000,0.0,0.0,0.0,0.0,0.0,0.0,3465,144,0


## 2. Fields: feature / label / context / excluded

***1. Feature Bucket:***
* **impressions_30d**: Total search impressions in the feature window (Feb 2026).
* **clicks_30d**: Total search clicks in the feature window.
* **ctr_30d**: Calculated click-through rate over the feature observation window.
* **avg_position**: Average search engine ranking position (Note: calculated as an unweighted daily average, which can directionally skew toward low-impression days).
* **ai_chatgpt_30d / ai_claude_30d / ai_gemini_30d**: Aggregated session traffic coming directly from AI assistants during the observation window (explicitly handled with zero-imputation via COALESCE).
* **word_count**: Actual word count joined directly from `dim_content`.
* **content_age_days**: Real operational age derived from actual publish dates.

***2. Label Bucket:***
* **is_declining**: Binary proxy indicator (1 if future impressions drop by 20% or more, or if content completely vanishes from the fact table in the subsequent window, 0 otherwise).

***3. Context Bucket:***
* **content_id**: Unique hashed identifier of the content page.
* **snapshot_date**: Date of the feature observation window.

***4. Excluded Bucket (and Why):***
* **post_snapshot_impressions_90d**: Excluded completely to prevent future window data leakage.
* **client workspace names / raw target URLs**: Excluded strictly for client privacy and safety compliance.

In [ ]:
feature_cols = [
    "impressions_30d",
    "clicks_30d",
    "ctr_30d",
    "avg_position",
    "ai_chatgpt_30d",
    "ai_claude_30d",
    "word_count",
    "content_age_days",
]
label_col = "is_declining"
context_cols = ["content_id", "snapshot_date"]

df[context_cols + feature_cols + [label_col]].head()

,content_id,snapshot_date,impressions_30d,clicks_30d,ctr_30d,avg_position,ai_chatgpt_30d,ai_claude_30d,word_count,content_age_days,is_declining
0,content_04c67f3541177192,2026-02-28,246.0,1.0,0.004065,151.464286,0.0,0.0,3168,144,0
1,content_05acc92c165f4386,2026-02-28,137.0,0.0,0.000000,49.037037,0.0,0.0,4135,144,1
2,content_0b33d8960857ad90,2026-02-28,8.0,0.0,0.000000,14.333333,0.0,0.0,3232,144,1
3,content_0f30e04e709c7b5d,2026-02-28,121.0,0.0,0.000000,39.428571,0.0,0.0,3211,144,0
4,content_1207efddce873942,2026-02-28,174.0,0.0,0.000000,194.000000,0.0,0.0,3465,144,0


## 3. Verify it with queries (grain, counts, missing values, windows)

**Verification Checks:**
* **Grain Check:** Confirm that there are no duplicate rows for the same content_id on each snapshot date.
* **Counts & Time Window:** Verify the total number of rows and confirm that the observation period is correct.
* **Data Availability / Missing Values:** Check data integrity where `is_available = TRUE`.

In [ ]:
# 1. Grain Check: Must return 0 duplicate content_ids
duplicates = duckdb.query("""
    SELECT content_id, COUNT(*) as cnt
    FROM df
    GROUP BY content_id
    HAVING COUNT(*) > 1
""").df()
print(f"1. Grain Check - Duplicate Rows Found: {len(duplicates)} (Expected: 0)")

# 2. Row Count & Snapshot Date Range
counts = duckdb.query("""
    SELECT
        COUNT(*) as total_rows,
        COUNT(DISTINCT content_id) as unique_pages,
        MIN(snapshot_date) as start_date,
        MAX(snapshot_date) as end_date
    FROM df
""").df()
print("\n2. Row Counts & Snapshot Window:")
print(counts)

# 3. Missing Value Analysis & Availability Integrity Check
availability = duckdb.query("""
    SELECT
        COUNT(*) as total_rows,
        COUNT(CASE WHEN is_available IS TRUE THEN 1 END) as available_rows,
        ROUND(AVG(CASE WHEN is_available IS TRUE THEN 1.0 ELSE 0.0 END) * 100, 2) as availability_pct
    FROM df
""").df()
print("\n3. Data Availability Verification:")
print(availability)

1. Grain Check - Duplicate Rows Found: 0 (Expected: 0)

2. Row Counts & Snapshot Window:
   total_rows  unique_pages  start_date    end_date
0      100000        100000  2026-02-28  2026-02-28

3. Data Availability Verification:
   total_rows  available_rows  availability_pct
0      100000          100000             100.0


## 4. Data limits
* **Unbalanced Panel History:** Client history depth varies across warehouse partitions.
* **Cold-Start Vulnerability:** Brand-new pages lack deep historical baselines, requiring dedicated handling during feature engineering.
* **AI-Referral Traffic Sparsity:** Only ~1.54% of content pages exhibit active AI referral traffic in this window, treated as sparse indicator signals.
* **Position Averaging Limitation:** `avg_position` utilizes an unweighted daily average which can directionally weight low-impression ranking spikes heavier than high-volume days.
* **Survivorship Bias Mitigation:** Addressed via a LEFT JOIN structure where pages completely missing from the target month's fact table are correctly marked as `is_declining = 1`.

In [ ]:
sparsity = duckdb.query("""
    SELECT
        COUNT(*) as total_pages,
        SUM(CASE WHEN content_age_days < 30 THEN 1 ELSE 0 END) as new_pages_count,
        ROUND(AVG(CASE WHEN content_age_days < 30 THEN 1.0 ELSE 0.0 END) * 100, 2) as cold_start_pct
    FROM df
""").df()

print("Boundary Verification (Real Cold Start / Sparse Pages):")
print(sparsity)

Boundary Verification (Real Cold Start / Sparse Pages):
   total_pages  new_pages_count  cold_start_pct
0       100000           9271.0            9.27


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.